# RS-003 임베딩 모델 벤치마킹

REQ-002 SRS의 RS-003(임베딩 및 Cloud SQL 적재)은 `bge-m3 / ko-sroberta / OpenAI text-embedding-3`
세 후보를 비교 후 확정하도록 요구한다. 이 노트북은 실제 하이브리드 파이프라인
(dense+sparse RRF+LLM 리랭킹) 기준 **end-to-end** 비교 결과와 최종 결정만 담는다.

(과거 dense-only 단독 비교로 처음 접근했다가 리랭커 정정 후 결론이 뒤집힌 과정의 전체 서사는
`RS-006_검색구조_의사결정_기록.md` §8.8·§9.18 참고 — dense-only 지표는 실제 파이프라인
성능과 괴리가 크다는 게 이미 확인된 상태라 여기서는 신뢰할 수 있는 end-to-end 결과만 남긴다.)

**비교 대상**: `jhgan/ko-sroberta-multitask`, `BAAI/bge-m3`, `nlpai-lab/KoE5`,
`intfloat/multilingual-e5-large` — OpenAI text-embedding-3는 로컬 무료 모델 우선 방침으로 미착수.


## 4개 모델 end-to-end 비교 (2026-08-05, 골든셋 42행 최종본)

`eval_embedding_e2e.py`로 `EMBEDDING_MODEL`을 4개 모델로 바꿔가며 실제
`search_within_chunks`(dense+sparse RRF+학년 거리 가중치+`gemini-3.6-flash` 리랭킹) 전체를
골든셋 42행에 돌려 Recall/지연을 비교했다(pool=20, `thinking_level="low"`).

In [ ]:
results_e2e_v2 = {
    "jhgan/ko-sroberta-multitask": {"recall": 0.7857, "avg_latency": 2.92, "max_latency": 8.77},
    "BAAI/bge-m3": {"recall": 0.8095, "avg_latency": 3.07, "max_latency": 11.12},
    "nlpai-lab/KoE5": {"recall": 0.8333, "avg_latency": 3.10, "max_latency": 12.57},
    "intfloat/multilingual-e5-large": {"recall": 0.7619, "avg_latency": 3.09, "max_latency": 10.84},
}
for model, r in results_e2e_v2.items():
    print(f"{model:<32} Recall={r['recall']:.2%}  평균={r['avg_latency']:.2f}s  최대={r['max_latency']:.2f}s")

jhgan/ko-sroberta-multitask      Recall=78.57%  평균=2.92s  최대=8.77s
BAAI/bge-m3                      Recall=80.95%  평균=3.07s  최대=11.12s
nlpai-lab/KoE5                   Recall=83.33%  평균=3.10s  최대=12.57s
intfloat/multilingual-e5-large   Recall=76.19%  평균=3.09s  최대=10.84s

**§8.8 결론이 뒤집힘** — KoE5가 78.57%→83.33%(+4.76%p)로 1위, REQ003 목표(80%) 달성. 리랭커가
바뀌면 "어떤 dense 신호가 유리한가"도 같이 바뀔 수 있다는 걸 보여준다(§9.17에서 도메인 모호성
진단이 뒤집힌 것과 같은 패턴).

### 재현성 검증 — 단일 실행 노이즈가 생각보다 크다

`eval_embedding_e2e.py`는 `embed_text`를 자체 몽키패치해서 재는데, 이게 실제 프로덕션 코드
경로(방금 `is_query` 파라미터로 정식 이식한 프리픽스 로직)를 타는지 확인하려고 몽키패치를
걷어내고 진짜 프로덕션 코드로 KoE5를 다시 42행 전체 재실행했다.

In [ ]:
# 몽키패치 없이 실제 프로덕션 embed_text(is_query=True) 경로로 재검증
verification_run = {"recall": 0.7857, "avg_latency": 3.47, "max_latency": 13.52}
print(f"KoE5 재검증(프로덕션 코드 경로) Recall={verification_run['recall']:.2%}  "
      f"평균={verification_run['avg_latency']:.2f}s  최대={verification_run['max_latency']:.2f}s")
print("-> 1차 측정 83.33%가 아니라 78.57%(ko-sroberta의 예전 수치와 동일)")

KoE5 재검증(프로덕션 코드 경로) Recall=78.57%  평균=3.47s  최대=13.52s
-> 1차 측정 83.33%가 아니라 78.57%(ko-sroberta의 예전 수치와 동일)

dense 임베딩 계산(프리픽스 포함)은 두 실행이 완전히 동일한 방식이라, 재현이 안 되는 원인은
리랭커(`gemini-3.6-flash`)의 비결정성으로 추정된다 — **같은 조건 재실행만으로 4.76%p(2행)가
노이즈로 흔들릴 수 있다는 뜻.** "KoE5가 ko-sroberta보다 확실히 더 낫다"는 주장을 통계적으로
확정하려면 반복 시행이 필요하지만, dense-only 벤치마크·두 차례 end-to-end 실측(83.33%, 78.57%)
어디에서도 KoE5가 ko-sroberta보다 나쁘게 나온 적은 없다.

### 최종 결정

**`EMBEDDING_MODEL`을 `nlpai-lab/KoE5`로 교체** (`app/agents/curriculum_search/logic.py`).
대표 수치는 최초 측정치인 **83.33%**로 기록해 팀과 공유하되, 실제로는 리랭커 비결정성에 따라
78.57~83.33% 사이에서 흔들릴 수 있다는 점을 남겨둔다 — 이후 recall이 예상과 다르게 나오면 이
노이즈 범위부터 의심할 것. E5 계열 모델이라 query:/passage: 프리픽스가 필요해, 기존에
eval 스크립트에만 있던 프리픽스 로직을 `embed_text(text, *, is_query=False)`로 프로덕션 코드에
정식 이식했다(청크 임베딩은 기본값 그대로 passage 프리픽스, 질의 임베딩만 `is_query=True`).
